In [1]:
import sys
rootdir_ = '../'
if ( rootdir_ not in sys.path ):
    sys.path.append(rootdir_)
    print( f" a path to {rootdir_} added in {__name__} ")

import netCDF4 as nc
import numpy as np
from scipy.sparse import csr_matrix

# The usual
from datetime import date
import xarray as xr
import matplotlib.pyplot as plt
import importlib
#import make_event_cubes as mec
from matplotlib.colors import SymLogNorm

from Utils import GridUtils as GrU
from Utils import utils as uti
from Utils import time_utils as tuti
from Utils import numerical_utils as nuti

import make_event_cubes as mec 
#importlib.reload(mec)


from pathlib import Path



# This allow both dict.key and dict['key'] syntax
class AttrDict(dict):
    def __getattr__(self, key):
        try:
            return self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")

    def __setattr__(self, key, value):
        self[key] = value

    def __delattr__(self, key):
        try:
            del self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")



def regrid(field_mpas, W, nlat, nlon):
    """
    field_mpas : np.ndarray, shape (n_src,) or (nz, n_src) or (nt, nz, n_src)
    Returns shape (..., nlat, nlon)
    """
    orig_shape = field_mpas.shape[:-1]
    flat = field_mpas.reshape(-1, field_mpas.shape[-1])  # (nfields, n_src)
    out  = (W @ flat.T).T                                 # (nfields, nlat*nlon)
    return out.reshape(orig_shape + (nlat, nlon))

#wgtfile='../mpasa3p75_TO_UHR_SO-Global_bilin.nc'
wgtfile='../mpasa3p75_TO_UHR_Lat_80S-0_Lon_0-360_bilin.nc'
DstScrip = '/glade/work/juliob/GridFiles/Scrip/latlon_UHR_Lat_80S-0_Lon_0-360_scrip.nc'


# Do this once at the top of your notebook/script
W, nlat, nlon, lat1d_hr, lon1d_hr = mec.make_regridder(wgtfile)


ftopo='/glade/campaign/cgd/amp/juliob/mpasa3p75km/cam77_dyamond1_prod1/TimeInvariant/PHIS_dyamond.nc'
Topo=xr.open_dataset( ftopo )
htopo=Topo.PHIS.values/9.8
htopo_yx=regrid(htopo, W, nlat, nlon)


start_date = [2016,8,1,3]
nsteps, step_size = 247,3

year,month,day,hour= start_date
start_date_a = f"{year:04d}-{month:02d}-{day:02d}-{hour*3_600:05d}"
dates=[]
for n in np.arange( nsteps ):
    date_=f"{year:04d}-{month:02d}-{day:02d}-{hour*3_600:05d}"
    dates.append( date_ )
    if day != 99:
        year,month,day,hour = tuti.increment_hours( [year,month,day,hour], nhours=step_size )


levels=[5_000,10_000,15_000,20_000]

for date in dates[0:3]:
    datadir = f"/glade/campaign/cgd/amp/juliob/mpasa3p75km/cam77_dyamond1_prod1/{date}"
    fields = ['U','V','U_prt','V_prt','PRECL','theta_mpas','w_mpas_prt',]
    frame_data_ = {'fields':fields, 'levels':levels, 'date':date, 'lon':lon1d_hr, 'lat':lat1d_hr, 'topo':htopo_yx }
    for fld in fields:
        file1 = f"{datadir}/{fld}_dyamond.nc"
        file2 = f"{datadir}/{fld}_dyamond.{date}.nc"
        fname = file1 if Path(file1).exists() else file2
        X = xr.open_dataset(fname)   
        print( f'opened {fname} {list(X.variables)} {X[fld].dims}') 
        if 'lev' in X[fld].dims:
            X = X.sel(lev=levels, method='nearest')
        elif 'ilev' in X[fld].dims:
            X = X.sel(ilev=levels, method='nearest')

        fld_c = X[fld].values 
        
        if ('lev' in X[fld].dims) or ('ilev' in X[fld].dims) :
            nt,nz,ncol = fld_c.shape
            fld_yx = np.zeros( (nt,nz,nlat,nlon) )
            for t in np.arange(nt):
                for z in np.arange(nz):
                    fld_yx[t,z,:,:] = regrid(fld_c[t,z,:], W, nlat, nlon)
        else:       
            nt,ncol = fld_c.shape
            fld_yx = np.zeros( (nt,nlat,nlon) )
            for t in np.arange(nt):
                fld_yx[t,:,:] = regrid(fld_c[t,:], W, nlat, nlon)

        frame_data_[fld] =  fld_yx
        
    frame_data =  AttrDict( frame_data_ )


    nt,nz,ny,nx = frame_data.U.shape
    zeta = np.zeros( ( nt,nz,ny,nx) )
    for t in np.arange(nt):
        for z in np.arange(nz):
            zeta[t,z,:,:] = nuti.Sphere_Curl2( f_x=frame_data.U[t,z,:,:] , f_y=frame_data.V[t,z,:,:] , lat=frame_data.lat , lon=frame_data.lon , wrap=True, verbose=False)
            print( z )
    
    frame_data['ZETA']=zeta
    
    sc=1.5
    fig,ax=plt.subplots( 1,1, figsize=(sc*16,sc*5) )
    #ax.contour( frame_data.lon , frame_data.lat, frame_data.topo,levels=[-500,1,10,100,1000,2000],cmap='terrain',alpha=0.25)
    #co = ax.contourf( frame_data.lon , frame_data.lat, frame_data.theta_mpas[0,0,:,:], levels=51,cmap='bwr' )  #, alpha=0.9 )
    #w_co_ = ax.contourf( cube['lon_hr'], cube['lat_hr'], cube['w_mpas_yx'][z,:,:], levels=wlevs, cmap='bwr' )
    #ax.contour( cube['lon_A'], cube['lat_A'], epwp[z,:,:] )
    #ax.set_xlim(40,180)
    
    lon,lat,data= frame_data.lon , frame_data.lat, frame_data.w_mpas_prt[0,3,:,:]
    
    norm = SymLogNorm(linthresh=1, vmin=-5, vmax=5)  # linthresh: where log kicks in
    
    #"""
    im = ax.imshow(data, origin='lower',
                   extent=[lon.min(), lon.max(), lat.min(), lat.max()],
                   #transform=ccrs.PlateCarree(),
                   cmap='bwr', norm=norm,
                   interpolation='nearest')
    """
    im = ax.imshow(data, origin='lower',
                   extent=[lon.min(), lon.max(), lat.min(), lat.max()],
                   #transform=ccrs.PlateCarree(),
                   cmap='bwr', vmin=-1, vmax=1,
                   interpolation='nearest')
    """
    
    plt.colorbar(im, ax=ax, orientation='vertical', shrink=0.6, label='w (m/s)')
    
    # --- topography overlay ---
    f = 4  # coarsening factor; try 2 or 8 to taste
    topo = frame_data.topo
    ny2 = (topo.shape[0] // f) * f
    nx2 = (topo.shape[1] // f) * f
    topo_c = topo[:ny2, :nx2].reshape(ny2//f, f, nx2//f, f).mean(axis=(1, 3))
    lat_c  = lat[:ny2].reshape(-1, f).mean(axis=1)
    lon_c  = lon[:nx2].reshape(-1, f).mean(axis=1)
    
    ax.contour(lon_c, lat_c, topo_c,
               levels=[1,1000, 2000, 3000, 4000],
               colors='k', linewidths=0.5, alpha=0.7)
    
    ax.set_xlabel('Longitude (°E)', fontsize=10)
    ax.set_ylabel('Latitude (°N)', fontsize=10)
    
    #plt.colorbar( co )
    fig.savefig(f'frame_{date}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)   # essential in a loop — otherwise figures pile up in memory

 a path to ../ added in __main__ 
 Utils.MyConstants in /glade/work/juliob/HiRes_ana_dev/Drivers/Utils 
Using Flexible parallel/serial VertRegrid 
 Utils.MyConstants in /glade/work/juliob/HiRes_ana_dev/Drivers/Utils 
 a path to /glade/work/juliob added in Utils.numerical_utils 
Regridder ready: 3600 lat x 12410 lon
  lat: -80.000 to 0.000 deg
  lon: 0.000 to 359.971 deg
opened /glade/campaign/cgd/amp/juliob/mpasa3p75km/cam77_dyamond1_prod1/2016-08-01-10800/U_dyamond.nc ['U', 'lat', 'lon', 'ilev', 'lev', 'time'] ('time', 'lev', 'ncol')
opened /glade/campaign/cgd/amp/juliob/mpasa3p75km/cam77_dyamond1_prod1/2016-08-01-10800/V_dyamond.nc ['V', 'lat', 'lon', 'ilev', 'lev', 'time'] ('time', 'lev', 'ncol')
opened /glade/campaign/cgd/amp/juliob/mpasa3p75km/cam77_dyamond1_prod1/2016-08-01-10800/U_prt_dyamond.2016-08-01-10800.nc ['lon', 'lat', 'U_prt', 'time', 'lev'] ('time', 'lev', 'ncol')
opened /glade/campaign/cgd/amp/juliob/mpasa3p75km/cam77_dyamond1_prod1/2016-08-01-10800/V_prt_dyamond.2016